# 18 · Data quality checks (Part D, step D3) — the *Veracity* evidence

**What this does:** measures how trustworthy every facility layer is, with Spark + Sedona, and writes one table for
the report. **Nothing is changed** — the cleaning itself was done when the layers were built (Parts A and B, the
health and fire builds) and in D1–D2; this notebook counts what is left and how each point was located.

**Checks** (each as rows flagged / rows checked):
- **missing values** — no location, no class range or students (schools), no type or district (health);
- **duplicates** — the same code, or the same name at the same point;
- **placeholder-like points** — one exact point shared by 4 or more facilities;
- **point outside its listed state** — a Sedona point-in-polygon join of every point with the 36 state polygons;
- **how points were located** — share of each location source and quality class (information, not errors);
- the D2 population-cell flags (coast cells given the nearest area, cells in no unit).

**Outputs:** `data/lake/gold/data_quality_report.csv` (all checks) and `data/lake/gold/data_quality_by_state.csv`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona, load the silver tables

In [4]:
import time

import pandas as pd
from pyspark.sql import functions as F
from bigdata.spark_session import get_spark
from build_india_health_facilities import fold  # the project's name folding (accents, case, state aliases)

SILVER = PROJECT / "data" / "lake" / "silver"
GOLD = PROJECT / "data" / "lake" / "gold"
GOLD.mkdir(parents=True, exist_ok=True)
spark = get_spark(PROJECT, memory="6g")
T0 = time.time()


def silver(name):
    return spark.read.format("geoparquet").load(str(SILVER / name))


schools = silver("schools_india").withColumn("_rid", F.monotonically_increasing_id()).cache()
health = silver("health_facilities").withColumn("_rid", F.monotonically_increasing_id()).cache()
kerala = silver("schools_kerala").withColumn("_rid", F.monotonically_increasing_id()).cache()
fire = silver("fire_stations").withColumn("_rid", F.monotonically_increasing_id()).cache()
states = silver("states")
states = states.join(F.broadcast(spark.createDataFrame([(n, fold(n)) for n in states.toPandas()["shapeName"]],
                                                       "shapeName string, geo_key string")), "shapeName")
print({n: t.count() for n, t in [("schools_india", schools), ("health", health), ("schools_kerala", kerala), ("fire", fire)]})
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

{'schools_india': 1070017, 'health': 228986, 'schools_kerala': 11764, 'fire': 129}
cores: 12 | Spark UI: http://127.0.0.1:4040


## Helpers

In [5]:
report = []


def add(dataset, check, flagged, total, note=""):
    report.append({"dataset": dataset, "check": check, "rows_checked": int(total), "rows_flagged": int(flagged),
                   "pct": round(100 * flagged / total, 3) if total else 0.0, "note": note})


def flag_counts(df, dataset, flags, note=None):
    """flags: {check name: boolean Spark expression}; one pass over the table."""
    total = df.count()
    r = df.agg(*[F.sum(F.when(F.expr(e), 1).otherwise(0)).alias(k) for k, e in flags.items()]).first()
    for k in flags:
        add(dataset, k, r[k] or 0, total, (note or {}).get(k, ""))
    return total


def shared_points(df, n=4):
    """Rows whose exact point is shared by n or more rows of the same table (a typical placeholder location)."""
    g = df.filter("geometry IS NOT NULL").groupBy("lat", "lon").count().filter(f"count >= {n}")
    return g.agg(F.sum("count")).first()[0] or 0


def duplicates(df, cols):
    """Rows that share all `cols` with another row (nulls ignored)."""
    g = df.dropna(subset=cols).groupBy(*cols).count().filter("count > 1")
    return g.agg(F.sum("count")).first()[0] or 0


def value_counts(df, dataset, col, label):
    total = df.count()
    for v, c in df.groupBy(col).count().orderBy(F.desc("count")).toPandas().itertuples(index=False):
        add(dataset, f"{label} = {v}", c, total, "share of rows, not an error")


def state_check(df, dataset, state_col):
    """Sedona point-in-polygon: is each point inside the state it is listed in? (36 state polygons broadcast)"""
    t = time.time()
    pts = df.filter("geometry IS NOT NULL").select("_rid", state_col, "geometry")
    keys = spark.createDataFrame([(s, fold(s)) for s in pts.select(state_col).distinct().toPandas()[state_col]],
                                 f"{state_col} string, listed_key string")
    hit = pts.alias("p").join(F.broadcast(states).alias("s"), F.expr("ST_Intersects(s.geometry, p.geometry)")) \
             .groupBy("p._rid").agg(F.collect_set("s.geo_key").alias("geo_keys"))
    j = pts.select("_rid", state_col).join(F.broadcast(keys), state_col, "left").join(hit, "_rid", "left") \
           .withColumn("status", F.when(F.col("geo_keys").isNull(), "in no state polygon")
                                  .when(F.array_contains("geo_keys", F.col("listed_key")), "ok")
                                  .otherwise("in another state")).cache()
    total = j.count()
    for s in ["in another state", "in no state polygon"]:
        add(dataset, f"point {s} (Sedona)", j.filter(F.col("status") == s).count(), total,
            "listed state vs geoBoundaries ADM1 polygon" + ("; usually on the coast or a border" if "no" in s else ""))
    pairs = j.filter("status = 'in another state'").select(state_col, F.concat_ws("+", "geo_keys").alias("point_in")) \
             .groupBy(state_col, "point_in").count().orderBy(F.desc("count")).limit(8).toPandas()
    print(f"{dataset}: state check on {total:,} points in {time.time() - t:.0f} s | most common mismatches:")
    print(pairs.to_string(index=False))
    j.unpersist()

## Schools, India

In [6]:
n = flag_counts(schools, "schools_india", {
    "no location": "geometry IS NULL",
    "no class range": "class_from IS NULL OR class_to IS NULL",
    "no student counts": "class_students IS NULL",
    "zero students": "class_students = 0",
    "no classrooms recorded": "class_rooms IS NULL OR class_rooms = 0",
    "no teachers recorded": "total_teachers IS NULL OR total_teachers = 0",
    "original UDISE point was a placeholder (fixed in Part B)": "placeholder",
    "original UDISE point was outside its state (fixed in Part B)": "outside_state",
    "original UDISE point far from its village (flag only)": "village_outlier",
    "point moved more than 1 km in Part B": "moved_m > 1000",
})
add("schools_india", "duplicate UDISE code", duplicates(schools, ["udise_code"]), n)
add("schools_india", "same name at the same point", duplicates(schools, ["school_name", "lat", "lon"]), n)
add("schools_india", "point shared by 4+ schools (after correction)", shared_points(schools), n,
    "village/cluster medians in Part B put several schools on one point")
value_counts(schools, "schools_india", "location_source", "location source")
value_counts(schools, "schools_india", "location_quality", "location quality")
state_check(schools, "schools_india", "state_name")

schools_india: state check on 1,070,016 points in 247 s | most common mismatches:
    state_name       point_in  count
        Kerala     puducherry      8
        Kerala      karnataka      3
 Uttar Pradesh madhya pradesh      3
   West Bengal          bihar      1
    Tamil Nadu     puducherry      1
Andhra Pradesh      karnataka      1


## Health facilities, India

In [7]:
n = flag_counts(health, "health_facilities", {
    "no location": "geometry IS NULL",
    "excluded from access analysis": "NOT in_access_analysis",
    "no facility type": "std_type IS NULL",
    "no district": "district IS NULL",
})
add("health_facilities", "same name at the same point", duplicates(health, ["name", "lat", "lon"]), n)
add("health_facilities", "point shared by 4+ facilities", shared_points(health), n)
for flag, c in (health.select(F.explode(F.split(F.coalesce("qa_flags", F.lit("")), "[; ]+")).alias("f"))
                .filter("f != ''").groupBy("f").count().orderBy(F.desc("count")).toPandas().itertuples(index=False)):
    add("health_facilities", f"QA flag from the build: {flag}", c, n, "set when the layer was built")
value_counts(health, "health_facilities", "location_quality", "location quality")
value_counts(health, "health_facilities", "location_from", "location from")
state_check(health, "health_facilities", "state")

health_facilities: state check on 228,230 points in 71 s | most common mismatches:
            state       point_in  count
      Uttarakhand  uttar pradesh    303
Jammu And Kashmir         ladakh    291
        Rajasthan madhya pradesh    147
   Andhra Pradesh      telangana    136
    Uttar Pradesh    uttarakhand    100
            Bihar  uttar pradesh     90
   Madhya Pradesh    maharashtra     81
        Rajasthan  uttar pradesh     76


## Schools, Kerala and fire stations

In [8]:
flags = {"no location": "geometry IS NULL", "no classrooms recorded": "classrooms IS NULL",
         "no teachers recorded": "teachers IS NULL", "no UDISE code": "udise_code IS NULL"}
for lvl in ["lp", "up", "hs", "hss"]:
    flags[f"teaches {lvl.upper()} but 0 students in it"] = f"has_{lvl} AND cap_{lvl} = 0"
n = flag_counts(kerala, "schools_kerala", flags)
add("schools_kerala", "duplicate UDISE code", duplicates(kerala, ["udise_code"]), n, "known: 8 codes shared by 2 Sametham schools")
add("schools_kerala", "point shared by 4+ schools", shared_points(kerala), n)
value_counts(kerala, "schools_kerala", "location_source", "location source")
state_check(kerala.withColumn("state", F.lit("Kerala")), "schools_kerala", "state")

n = flag_counts(fire, "fire_stations", {"no location": "geometry IS NULL",
                                        "nearest other station under 1 km": "nearest_station_m < 1000"})
add("fire_stations", "point shared by 2+ stations", shared_points(fire, 2), n)
state_check(fire, "fire_stations", "state")

schools_kerala: state check on 11,763 points in 23 s | most common mismatches:
 state   point_in  count
Kerala puducherry      8
Kerala  karnataka      3


fire_stations: state check on 129 points in 19 s | most common mismatches:
Empty DataFrame
Columns: [state, point_in, count]
Index: []


## Population cells (from D2)

In [9]:
cells = spark.read.parquet(str(SILVER / "population_cells"))
r = cells.agg(F.count("*").alias("n"), F.sum(F.col("tag_nearest").cast("int")).alias("nearest"),
              F.sum(F.when(F.col("state") == "(outside boundaries)", 1).otherwise(0)).alias("no_state"),
              F.sum(F.when(F.col("district_code") == 0, 1).otherwise(0)).alias("no_district"),
              F.sum(F.when(F.col("subdistrict_code") == 0, 1).otherwise(0)).alias("no_subdistrict"),
              F.sum(F.when((F.col("state") == "Kerala") & (F.col("local_body_code") == 0), 1).otherwise(0)).alias("kerala_no_lb"),
              F.sum(F.col("age_nodata").cast("int")).alias("age_nodata")).first()
for k, label in [("nearest", "area given by nearest polygon (coast/gaps)"), ("no_state", "in no state"),
                 ("no_district", "in no district"), ("no_subdistrict", "in no sub-district"),
                 ("kerala_no_lb", "Kerala cell in no OSM local body"), ("age_nodata", "age band missing")]:
    add("population_cells", label, r[k], r["n"], "from step D2")

## Main checks by state

In [10]:
# The same main checks by state, for the report's state table
def by_state(df, state_col, dataset):
    shared = df.filter("geometry IS NOT NULL").groupBy("lat", "lon").count().filter("count >= 4").select("lat", "lon")
    flagged = df.join(shared.withColumn("shared4", F.lit(1)), ["lat", "lon"], "left")
    return flagged.groupBy(F.col(state_col).alias("state")).agg(
        F.count("*").alias("rows"), F.sum(F.col("geometry").isNull().cast("int")).alias("no_location"),
        F.sum(F.coalesce("shared4", F.lit(0))).alias("point_shared_4plus")).withColumn("dataset", F.lit(dataset)).toPandas()


bs = pd.concat([by_state(schools, "state_name", "schools_india"), by_state(health, "state", "health_facilities")])
bs["pct_shared_4plus"] = (100 * bs["point_shared_4plus"] / bs["rows"]).round(2)
bs = bs.sort_values(["dataset", "rows"], ascending=[True, False])
bs.to_csv(GOLD / "data_quality_by_state.csv", index=False)
print(bs.head(12).to_string(index=False))

         state  rows  no_location  point_shared_4plus           dataset  pct_shared_4plus
 Uttar Pradesh 32387           86                1442 health_facilities              4.45
     Rajasthan 19773            1                2940 health_facilities             14.87
   Maharashtra 18911            0                1237 health_facilities              6.54
Madhya Pradesh 15471            2                 314 health_facilities              2.03
     Karnataka 14246            0                 374 health_facilities              2.63
    Tamil Nadu 13912            1                 335 health_facilities              2.41
   West Bengal 12597            4                  30 health_facilities              0.24
         Bihar 12129          102                1772 health_facilities             14.61
Andhra Pradesh 11984            0                1443 health_facilities             12.04
       Gujarat 10934           22                 113 health_facilities              1.03
        Od

## Report

In [11]:
rep = pd.DataFrame(report)
rep.to_csv(GOLD / "data_quality_report.csv", index=False)
pd.set_option("display.width", 200)
print(rep.to_string(index=False, max_colwidth=70))
print(f"\nwritten {GOLD / 'data_quality_report.csv'} ({len(rep)} checks) and data_quality_by_state.csv "
      f"| total time {(time.time() - T0) / 60:.1f} min")
spark.stop()

          dataset                                                        check  rows_checked  rows_flagged    pct                                                                   note
    schools_india                                                  no location       1070017             1  0.000                                                                       
    schools_india                                               no class range       1070017           392  0.037                                                                       
    schools_india                                            no student counts       1070017           392  0.037                                                                       
    schools_india                                                zero students       1070017          4094  0.383                                                                       
    schools_india                                       no classrooms recor